In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from torch.utils.data import DataLoader

from model import NNUE
from dataset import transform_row, transform_batch, nnue_collate_fn

import os
import glob

/Users/andyxiong/Desktop/USC/year 2/535 multimodal/535env_arm64/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(device)

mps


In [3]:
ds = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train", 
    streaming=True)

original_columns = ds.column_names
transformed_ds = ds.map(
    transform_row,
    remove_columns=original_columns
).filter(lambda x: x is not None)

In [4]:
# Take a small sample and manually iterate
sample = transformed_ds.take(100)
for i, item in enumerate(sample):
    if item is None:
        print(f"Row {i} is None!")
    # If item is a dict, check the values
    elif isinstance(item, dict):
        for k, v in item.items():
            if v is None:
                print(f"Row {i} has None in key: {k}")

In [5]:
# Initialize Model, Loss, and Optimizer
model = NNUE().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW([
    {'params': model.feature_transformer.parameters(), 'weight_decay': 0.0},
    {'params': model.output_layer.parameters(), 'weight_decay': 1e-4}
], lr=1e-3, betas=(0.9, 0.999), eps=1e-8)

#checkpoint = torch.load(f"nnue_checkpoints/chess_model_final.pt")
#model.load_state_dict(checkpoint["model_state_dict"])
#optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [6]:
# Setup Splitting
shuffled_ds = transformed_ds.shuffle(seed=42, buffer_size=10000)

val_ds = shuffled_ds.take(2048) 
train_ds = shuffled_ds.skip(2048 + 1024 * 0)

val_loader = DataLoader(val_ds, batch_size=1024, num_workers=0, collate_fn=nnue_collate_fn)
pin_memory = device == torch.device('cuda')
train_loader = DataLoader(train_ds, batch_size=4096, num_workers=0, collate_fn=nnue_collate_fn, pin_memory=pin_memory)

In [7]:
# Training Hyperparameters
NUM_EPOCHS = 3
STEPS_PER_EPOCH = 50000  # set based on dataset size / batch size
VAL_INTERVAL = 2000  # Validate every 2000 batches
SAVE_INTERVAL = 10000  # Save checkpoint

In [8]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    total_steps=STEPS_PER_EPOCH * NUM_EPOCHS,
    pct_start=0.05,      # 5% of steps spent warming up
    anneal_strategy='cos',
    final_div_factor=100  # ends at 1e-5
)

In [11]:
def find_latest_checkpoint(checkpoint_dir="nnue_checkpoints"):
    all_ckpts = (glob.glob(f"{checkpoint_dir}/chess_model_step_*.pt") +
                 glob.glob(f"{checkpoint_dir}/chess_model_epoch_*.pt"))
    if not all_ckpts:
        return None
    return max(all_ckpts, key=lambda p: torch.load(p, map_location='cpu').get('step', 0))

RESUME_FROM = find_latest_checkpoint()

start_epoch = 0
global_step = 0

if RESUME_FROM and os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])

    global_step = ckpt['step']
    saved_epoch = ckpt['epoch']
    is_epoch_end = ckpt.get('is_epoch_end', False)
    # Mid-epoch checkpoint: restart that epoch from scratch; epoch-end: start the next one
    start_epoch = saved_epoch + 1 if is_epoch_end else saved_epoch

    print(f"Resumed '{RESUME_FROM}': starting epoch {start_epoch + 1}/{NUM_EPOCHS}, global_step={global_step}")
else:
    print("No checkpoint found — starting fresh")

No checkpoint found — starting fresh


In [ ]:
for epoch in range(start_epoch, NUM_EPOCHS):
    train_loader = DataLoader(train_ds, batch_size=4096, num_workers=0, collate_fn=nnue_collate_fn, pin_memory=pin_memory)

    model.train()
    pbar = tqdm(enumerate(train_loader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", smoothing=0)

    running_train_loss = 0.0
    train_samples_since_val = 0

    for step, batch in pbar:
        stm_idx = batch['stm_idx'].to(device)
        stm_off = batch['stm_off'].to(device)
        nstm_idx = batch['nstm_idx'].to(device)
        nstm_off = batch['nstm_off'].to(device)
        targets = batch['target'].to(device)
        batch_size = targets.size(0)

        outputs = model(stm_idx, stm_off, nstm_idx, nstm_off)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_train_loss += loss.item() * batch_size
        train_samples_since_val += batch_size
        global_step += 1

        # --- Validation Loop ---
        if global_step % VAL_INTERVAL == 0:
            avg_train_loss = running_train_loss / train_samples_since_val

            model.eval()
            val_loss = 0.0
            total_val_samples = 0

            with torch.no_grad():
                for v_batch in val_loader:
                    v_stm_idx = v_batch['stm_idx'].to(device)
                    v_stm_off = v_batch['stm_off'].to(device)
                    v_nstm_idx = v_batch['nstm_idx'].to(device)
                    v_nstm_off = v_batch['nstm_off'].to(device)
                    vt = v_batch['target'].to(device)

                    current_batch_size = vt.size(0)
                    v_out = model(v_stm_idx, v_stm_off, v_nstm_idx, v_nstm_off)
                    v_loss = criterion(v_out, vt)
                    val_loss += v_loss.item() * current_batch_size
                    total_val_samples += current_batch_size

            if total_val_samples > 0:
                avg_val_loss = val_loss / total_val_samples
                print(f"\n[Epoch {epoch+1} | Step {global_step}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
            else:
                print(f"\n[Epoch {epoch+1} | Step {global_step}] Validation produced no samples.")

            if device == torch.device('mps'):
                torch.mps.empty_cache()

            running_train_loss = 0.0
            train_samples_since_val = 0
            model.train()

        # --- Checkpointing ---
        if global_step % SAVE_INTERVAL == 0:
            save_loc = f"nnue_checkpoints/chess_model_step_{global_step}.pt"
            torch.save({
                'epoch': epoch,
                'step': global_step,
                'is_epoch_end': False,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': loss.item(),
            }, save_loc)
            print(f"Saved model to {save_loc}")

    # Save at end of each epoch
    save_loc = f"nnue_checkpoints/chess_model_epoch_{epoch+1}.pt"
    torch.save({
        'epoch': epoch,
        'step': global_step,
        'is_epoch_end': True,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': loss.item(),
    }, save_loc)
    print(f"\nEpoch {epoch+1} complete. Saved to {save_loc}")

Epoch 1/3: 2001it [07:53,  4.22it/s]


[Epoch 1 | Step 2000] Train Loss: 0.22068 | Val Loss: 0.32976


Epoch 1/3: 4000it [16:54,  3.94it/s]


[Epoch 1 | Step 4000] Train Loss: 0.22302 | Val Loss: 0.28683


Epoch 1/3: 6000it [27:47,  3.60it/s]

...

Epoch 1/3: 20000it [2:14:03,  2.49it/s]

Saved model to nnue_checkpoints/chess_model_step_20000.pt


Epoch 1/3: 22000it [2:27:16,  2.49it/s]


[Epoch 1 | Step 22000] Train Loss: 0.16493 | Val Loss: 0.29951


Epoch 1/3: 22921it [2:33:24,  2.49it/s]

In [ ]:
torch.save({
    'step': step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss.item(),
}, f"nnue_checkpoints/chess_model_final.pt")